# Python for Data Analysis, chapter 7: data cleaning and preparation

The most useful examples from the chapter, rewritten for **pandas 3**, on a messy support-ticket export like the ones you see at work.

**How to use this notebook**
1. Run each cell with **Shift+Enter** and read the output before moving on.
2. Change something in each example and guess the result before running it.
3. The **Your turn** tasks at the end use the real GitHub incidents file. Write those yourself.

Book chapter: https://wesmckinney.com/book/data-cleaning

In [ ]:
import io
import json
from pathlib import Path

import numpy as np
import pandas as pd

SAMPLES = Path("../samples")

## The messy data

Look at the raw text before running anything. Spot the problems yourself first: missing values written three different ways, a duplicated row, inconsistent team names, durations with units, an impossible value, a typo.

In [ ]:
raw_csv = """ticket_id,opened,team,priority,reporter_email,time_to_close,description
101,2026-11-01 08:15,Network,high,dana@ness.co.il,45 min,VPN down for floor 3
102,2026-11-01 09:30,support ,Low,avi@ness.co.il,N/A,Printer jam
103,2026-11-02 11:00,SUPPORT,medium,noa@client.com,2h,Outlook crash ERR-4021 after update
103,2026-11-02 11:00,SUPPORT,medium,noa@client.com,2h,Outlook crash ERR-4021 after update
104,2026-11-02 14:20,network,hgih,,90 min,Switch port flapping
105,2026-11-03 10:05,Data,low,omer@ness.co.il,-,Report refresh failed ERR-1102
106,2026-11-03 16:40,support,high,maya@client.com,9999 min,Laptop won't boot
107,2026-11-04 08:00,Network,,lior@ness.co.il,30 min,Wi-Fi slow ERR-0007
"""

tickets = pd.read_csv(io.StringIO(raw_csv))
tickets

## 1. Missing data

`read_csv` already turned the empty cells and `N/A` into missing values, but not `-`. Tell it which texts mean "missing" with `na_values`.

In [ ]:
tickets = pd.read_csv(io.StringIO(raw_csv), na_values=["N/A", "-", ""])
tickets.isna().sum()     # missing values per column

In [ ]:
# Rows with any missing value
tickets[tickets.isna().any(axis=1)]

In [ ]:
# Three ways to deal with them; each returns a new table, `tickets` is unchanged
print(len(tickets.dropna()), "rows if we drop every row with a gap")
print(len(tickets.dropna(subset=["priority"])), "rows if we only require a priority")
tickets.fillna({"priority": "unknown", "reporter_email": "unknown"}).head(3)

Which one is right depends on the question. Dropping rows silently removes data, so in a pipeline you usually **count** the gaps and keep a record, rather than deleting them.

## 2. Duplicates

In [ ]:
tickets.duplicated()     # True for a row that repeats an earlier one exactly

In [ ]:
tickets = tickets.drop_duplicates()
len(tickets)

In [ ]:
# Duplicates by key only: same ticket ID, even if other columns differ
tickets.duplicated(subset=["ticket_id"]).sum()

## 3. Cleaning text: the `.str` accessor

`.str` applies a string method to every value in a column at once. No loops.

In [ ]:
tickets["team"].unique()     # 'Network', 'support ', 'SUPPORT', 'network', 'Data'...

In [ ]:
tickets["team"] = tickets["team"].str.strip().str.lower()
tickets["team"].value_counts()

In [ ]:
tickets["priority"] = tickets["priority"].str.lower()
tickets["priority"].unique()     # still has the typo 'hgih'

## 4. Replacing values: `replace` and `map`

`replace` changes only the values you list. `map` translates every value through a dictionary; anything not in it becomes missing.

In [ ]:
tickets["priority"] = tickets["priority"].replace({"hgih": "high"})
tickets["priority"].value_counts(dropna=False)

In [ ]:
# map: turn a label into a number for sorting
priority_rank = {"high": 1, "medium": 2, "low": 3}
tickets["priority_rank"] = tickets["priority"].map(priority_rank)
tickets[["ticket_id", "priority", "priority_rank"]]

## 5. Regular expressions: pulling structure out of text

`str.extract` keeps the part of the text inside the brackets of a pattern. Two of the most useful patterns for logs and tickets:

In [ ]:
# The error code, if the description has one: ERR- followed by 4 digits
tickets["error_code"] = tickets["description"].str.extract(r"(ERR-\d{4})", expand=False)
tickets[["ticket_id", "description", "error_code"]]

In [ ]:
# The email domain: everything after the @
tickets["domain"] = tickets["reporter_email"].str.extract(r"@(.+)$", expand=False)
tickets["domain"].value_counts(dropna=False)

## 6. Turning text into numbers

`time_to_close` mixes units: `45 min`, `2h`. Extract the number and the unit with one pattern, then convert everything to minutes.

In [ ]:
parts = tickets["time_to_close"].str.extract(r"(\d+)\s*(min|h)")
parts.columns = ["amount", "unit"]
parts

In [ ]:
amount = pd.to_numeric(parts["amount"])
tickets["minutes_to_close"] = amount.where(parts["unit"] == "min", amount * 60)
tickets[["ticket_id", "time_to_close", "minutes_to_close"]]

## 7. Outliers

9999 minutes is almost a week for a laptop that won't boot. Probably a placeholder someone typed, not a real duration. Find values outside a sensible range and decide what to do with them: here we mark them as missing rather than delete the ticket.

In [ ]:
tickets["minutes_to_close"].describe()

In [ ]:
too_long = tickets["minutes_to_close"] > 24 * 60          # more than a day
tickets.loc[too_long, ["ticket_id", "minutes_to_close"]]

In [ ]:
tickets.loc[too_long, "minutes_to_close"] = np.nan
tickets["minutes_to_close"].describe()

## 8. Binning: `pd.cut`

Turn a number into ranges, such as SLA buckets.

In [ ]:
tickets["sla_bucket"] = pd.cut(
    tickets["minutes_to_close"],
    bins=[0, 60, 240, np.inf],
    labels=["under 1h", "1-4h", "over 4h"],
)
tickets[["ticket_id", "minutes_to_close", "sla_bucket"]]

## 9. Types: dates, nullable integers and categories

In [ ]:
tickets["opened"] = pd.to_datetime(tickets["opened"])
tickets["minutes_to_close"] = tickets["minutes_to_close"].astype("Int64")   # whole numbers, gaps allowed
tickets["team"] = tickets["team"].astype("category")                       # few repeating labels
tickets.dtypes

## 10. Renaming columns

In [ ]:
clean = tickets.rename(columns={"reporter_email": "reporter", "time_to_close": "time_to_close_raw"})
clean.columns.tolist()

## Your turn

Use the real file `github_incidents.json`. Write these yourself; each needs a few lines.

1. Load it into `incidents` with `json_normalize`, as in chapter 6. How many missing values does each column have? Which columns are missing in **every** row?
2. Are there duplicate incident IDs? Show the count.
3. Make a column `name_clean`: the incident `name` in lowercase with spaces trimmed. Then count how many names contain the word `actions` (hint: `.str.contains`).
4. With `str.extract`, pull the first word after `"with "` from each name, for example `"Incident with Git Operations"` gives `Git`. How many incidents mention each one?
5. Compute `hours_to_resolve`, then bin it with `pd.cut` into `under 1h`, `1-4h`, `over 4h`. How many incidents fall in each bucket?
6. Make `impact` a `category` column with the order `none < minor < major < critical`, and sort the incidents by it, most severe first. (Hint: `pd.CategoricalDtype([...], ordered=True)`.)

## Notes in my own words

*(3–4 lines: the difference between `dropna` and `fillna`, what `.str` does, when to use `replace` vs `map`, and why you mark outliers instead of deleting rows.)*